# **Intent Classification using Groq API**

### Importing Libraries

In [28]:
import subprocess
import sys
import tempfile
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from dotenv import load_dotenv

#### We will be experimenting with two types of Prompt & two diff Models:
##### Prompt Type:
- Zero Shots Prompting
- Few Shots Prompting
##### Model:
- LlaMA 4 Model
- GPT OSS

In [29]:
IntentClassificationPrompt_ZeroShots = f"""
You are a helpful assistant specialized in intent classification.
There are 5 classes to be classified into:

1- greeting
2- goodbye
3- gratitude
4- asking_mental_health_question
5- out_of_scope

Your output should only be the class name without any additional text or explanation.
Never output more than one class. If the text does not fit into any of the above classes, classify it as "out_of_scope".

Please classify the intent of the following text:
{{text}}
"""

In [30]:

IntentClassificationPrompt_FewShots = f"""
You are a helpful assistant specialized in intent classification.
There are 5 classes to be classified into:

1- greeting
2- goodbye
3- gratitude
4- asking_mental_health_question
5- out_of_scope

Your output should only be the class name without any additional text or explanation.
Never output more than one class. If the text does not fit into any of the above classes, classify it as "out_of_scope".

Take the following examples as a reference for how to classify the intents:

Examples:

1) 
Input: "Hello, how are you?"
Output: "greeting"

2)
Input: "Thank you for your help!"
Output: "gratitude"

3)
Input: "Can you tell me about anxiety?"
Output: "asking_mental_health_question"

4)
Input: "Okay! Goodbye!"
Output: "goodbye"

5)
Input: "Who won the World Cup in 2022?"
Output: "out_of_scope"


Please classify the intent of the following text:
{{text}}
"""

#### Export the API_KEY in the .env file!

In [31]:

class IntentClassifier:
    def __init__(self, model):
        self.model = model
        load_dotenv()
        try:
            self.API_KEY = os.getenv("GROQ_API_KEY")
            if not self.API_KEY:
                raise ValueError("GROQ_API_KEY not found in environment variables.")
        except Exception as e:
            print(f"Error loading API key: {e}")
            sys.exit(1)
            
        self.initialize_LLM()
    
    def initialize_LLM(self):
        self.llm = ChatGroq(
            model=self.model,
            temperature=0.5,
            max_tokens=None,
            timeout=None,
            max_retries=5,
            api_key=self.API_KEY
        )
        
    def classify(self, text, prompt):
        prompt = prompt.format(text=text)
        generated_text = self.llm.invoke([HumanMessage(content=prompt)])
        return generated_text.content.strip()

### Populating a list of test cases

In [ ]:
import pandas as pd

classifier_LlaMA = IntentClassifier("meta-llama/llama-4-scout-17b-16e-instruct")
classifier_GPTOSS = IntentClassifier("openai/gpt-oss-120b")

test_sentences = [
    ("Hello, how are you?",                                             "greeting"),
    ("Hey there! Good morning!",                                        "greeting"),
    ("Hi! How's everything going?",                                     "greeting"),
    ("Good evening, how are you doing?",                                "greeting"),
    ("Howdy! Nice to meet you.",                                        "greeting"),
    ("Good afternoon! Hope you're having a great day.",                 "greeting"),
    ("Hey doc, how are you?",                                           "greeting"),
    ("Hi there, hope you're well!",                                     "greeting"),

    ("Okay! Goodbye!",                                                  "goodbye"),
    ("See you later, take care!",                                       "goodbye"),
    ("Bye bye, talk soon.",                                             "goodbye"),
    ("I have to go now, farewell.",                                     "goodbye"),
    ("It was nice talking to you, goodbye!",                            "goodbye"),
    ("I'm heading out, see you around.",                                "goodbye"),
    ("Goodnight, take care of yourself.",                               "goodbye"),
    ("Alright, I'm signing off now.",                                   "goodbye"),

    ("Thank you for your help!",                                        "gratitude"),
    ("I really appreciate everything you've done for me.",              "gratitude"),
    ("Thanks a lot, you're amazing!",                                   "gratitude"),
    ("I have missed you so much!",                                      "gratitude"),
    ("You've been so kind, thank you.",                                 "gratitude"),
    ("Many thanks for being there for me.",                             "gratitude"),
    ("You're the best, truly appreciate it.",                           "gratitude"),
    ("Thanks for listening, it means a lot.",                           "gratitude"),

    ("Can you tell me about anxiety?",                                  "asking_mental_health_question"),
    ("Hey doc, I've been feeling incredibly overwhelmed and sad lately.","asking_mental_health_question"),
    ("How are you doing today?",                                        "greeting"),  # tricky: looks like MH but it's a greeting
    ("I've been having panic attacks, what should I do?",               "asking_mental_health_question"),
    ("What are the symptoms of depression?",                            "asking_mental_health_question"),
    ("I feel very lonely and disconnected from everyone around me.",    "asking_mental_health_question"),
    ("How do I cope with grief after losing someone?",                  "asking_mental_health_question"),
    ("Is it normal to feel numb and empty all the time?",               "asking_mental_health_question"),

    ("Who won the World Cup in 2022?",                                  "out_of_scope"),
    ("What is the weather like in Cairo?",                              "out_of_scope"),
    ("How do I bake a chocolate cake?",                                 "out_of_scope"),
    ("Can you recommend a good sci-fi book?",                           "out_of_scope"),
    ("How does a black hole form?",                                     "out_of_scope"),
    ("What is the latest iPhone model?",                                "out_of_scope"),
    ("Who wrote the Harry Potter series?",                              "out_of_scope"),
]

In [33]:
results_LlaMA = []
results_GPTOSS = []

print("Running classifications... (Generating side-by-side view)")

for text in test_sentences:
    ground_truth = text[1]
    text = text[0] 
    print(f"Processing: {text}")
    zero_shot_res_LlaMA = classifier_LlaMA.classify(text, IntentClassificationPrompt_ZeroShots)
    few_shot_res_LlaMA = classifier_LlaMA.classify(text, IntentClassificationPrompt_FewShots)
    
    zero_shot_res_GPTOSS = classifier_GPTOSS.classify(text, IntentClassificationPrompt_ZeroShots)
    few_shot_res_GPTOSS = classifier_GPTOSS.classify(text, IntentClassificationPrompt_FewShots)
    print("ground_truth:", ground_truth)
    results_LlaMA.append({
        "Test Input": text,
        "Zero-Shot Prediction": zero_shot_res_LlaMA,
        "Few-Shot Prediction": few_shot_res_LlaMA,
        "Matches?": zero_shot_res_LlaMA == few_shot_res_LlaMA,
        "ground_truth": ground_truth,
        "Zero-Shot Correct?": zero_shot_res_LlaMA == ground_truth,
        "Few-Shot Correct?": few_shot_res_LlaMA == ground_truth
    })
    
    results_GPTOSS.append({
        "Test Input": text,
        "Zero-Shot Prediction": zero_shot_res_GPTOSS,
        "Few-Shot Prediction": few_shot_res_GPTOSS,
        "Matches?": zero_shot_res_GPTOSS == few_shot_res_GPTOSS,
        "ground_truth": ground_truth,
        "Zero-Shot Correct?": zero_shot_res_GPTOSS == ground_truth,
        "Few-Shot Correct?": few_shot_res_GPTOSS == ground_truth
    })

Running classifications... (Generating side-by-side view)
Processing: Hello, how are you?
ground_truth: greeting
Processing: Hey there! Good morning!
ground_truth: greeting
Processing: Hi! How's everything going?
ground_truth: greeting
Processing: Good evening, how are you doing?
ground_truth: greeting
Processing: Howdy! Nice to meet you.
ground_truth: greeting
Processing: Good afternoon! Hope you're having a great day.
ground_truth: greeting
Processing: Hey doc, how are you?
ground_truth: greeting
Processing: Hi there, hope you're well!
ground_truth: greeting
Processing: Okay! Goodbye!
ground_truth: goodbye
Processing: See you later, take care!
ground_truth: goodbye
Processing: Bye bye, talk soon.
ground_truth: goodbye
Processing: I have to go now, farewell.
ground_truth: goodbye
Processing: It was nice talking to you, goodbye!
ground_truth: goodbye
Processing: I'm heading out, see you around.
ground_truth: goodbye
Processing: Goodnight, take care of yourself.
ground_truth: goodbye
Pr

In [ ]:
df_LlaMA = pd.DataFrame(results_LlaMA)
df_GPTOSS = pd.DataFrame(results_GPTOSS)

### Display the full table for LlaMA

In [35]:
display(df_LlaMA)

,Test Input,Zero-Shot Prediction,Few-Shot Prediction,Matches?,ground_truth,Zero-Shot Correct?,Few-Shot Correct?
0,"Hello, how are you?",greeting,greeting,True,greeting,True,True
1,Hey there! Good morning!,greeting,greeting,True,greeting,True,True
2,Hi! How's everything going?,greeting,greeting,True,greeting,True,True
3,"Good evening, how are you doing?",greeting,greeting,True,greeting,True,True
4,Howdy! Nice to meet you.,greeting,greeting,True,greeting,True,True
5,Good afternoon! Hope you're having a great day.,greeting,greeting,True,greeting,True,True
6,"Hey doc, how are you?",greeting,greeting,True,greeting,True,True
7,"Hi there, hope you're well!",greeting,greeting,True,greeting,True,True
8,Okay! Goodbye!,goodbye,goodbye,True,goodbye,True,True
9,"See you later, take care!",goodbye,goodbye,True,goodbye,True,True


### Display the full table for GPT_OSS

In [36]:
display(df_GPTOSS)

,Test Input,Zero-Shot Prediction,Few-Shot Prediction,Matches?,ground_truth,Zero-Shot Correct?,Few-Shot Correct?
0,"Hello, how are you?",greeting,greeting,True,greeting,True,True
1,Hey there! Good morning!,greeting,greeting,True,greeting,True,True
2,Hi! How's everything going?,greeting,greeting,True,greeting,True,True
3,"Good evening, how are you doing?",greeting,greeting,True,greeting,True,True
4,Howdy! Nice to meet you.,greeting,greeting,True,greeting,True,True
5,Good afternoon! Hope you're having a great day.,greeting,greeting,True,greeting,True,True
6,"Hey doc, how are you?",greeting,greeting,True,greeting,True,True
7,"Hi there, hope you're well!",greeting,greeting,True,greeting,True,True
8,Okay! Goodbye!,goodbye,goodbye,True,goodbye,True,True
9,"See you later, take care!",goodbye,goodbye,True,goodbye,True,True


### Show ONLY rows where Zero-Shot and Few-Shot came up with different answers (LlaMA)

In [37]:
mismatches_LlaMA = df_LlaMA[df_LlaMA["Matches?"] == False]
print(f"Total Discrepancies Found: {len(mismatches_LlaMA)}")
if len(mismatches_LlaMA) > 0:
    display(mismatches_LlaMA)

Total Discrepancies Found: 0


### Show ONLY rows where Zero-Shot and Few-Shot came up with different answers (GPT_OSS)

In [ ]:
for model_name, df_model in [("LlaMA", df_LlaMA), ("GPT OSS", df_GPTOSS)]:
    sub = df_model[["Test Input", "Zero-Shot Prediction", "Few-Shot Prediction", "Matches?", "Zero-Shot Correct?", "Few-Shot Correct?"]].copy()
    sub.columns = ["Input", "Zero-shot pred", "Few-shot pred", "Matches?", "Zero-Shot Correct?", "Few-Shot Correct?"]

    def row_style(row):
        if row["Matches?"]:
            bg = ""
        else:
            bg = "background-color:#3d2e10"
        return [bg] * len(row)
    def row_style2(row):
        if row["Zero-Shot Correct?"] and row["Few-Shot Correct?"]:
            bg = "background-color:#1a4d2e"
        elif row["Zero-Shot Correct?"] and not row["Few-Shot Correct?"]:
            bg = "background-color:#4d2e1a"
        elif not row["Zero-Shot Correct?"] and row["Few-Shot Correct?"]:
            bg = "background-color:#4d1a2e"
        else:
            bg = "background-color:#3d2e10"
        return [bg] * len(row)

    styled = (
        sub.style
           .apply(row_style, axis=1)
           .apply(row_style2, axis=1)
           .set_caption(f"── {model_name} ──")
           .set_table_styles([{"selector":"caption","props":[("font-size","15px"),
                                                              ("font-weight","bold"),
                                                              ("text-align","left"),
                                                              ("padding","8px 0")]}])
    )
    display(styled)
    print()


,Input,Zero-shot pred,Few-shot pred,Matches?,Zero-Shot Correct?,Few-Shot Correct?
0,"Hello, how are you?",greeting,greeting,True,True,True
1,Hey there! Good morning!,greeting,greeting,True,True,True
2,Hi! How's everything going?,greeting,greeting,True,True,True
3,"Good evening, how are you doing?",greeting,greeting,True,True,True
4,Howdy! Nice to meet you.,greeting,greeting,True,True,True
5,Good afternoon! Hope you're having a great day.,greeting,greeting,True,True,True
6,"Hey doc, how are you?",greeting,greeting,True,True,True
7,"Hi there, hope you're well!",greeting,greeting,True,True,True
8,Okay! Goodbye!,goodbye,goodbye,True,True,True
9,"See you later, take care!",goodbye,goodbye,True,True,True


,Input,Zero-shot pred,Few-shot pred,Matches?,Zero-Shot Correct?,Few-Shot Correct?
0,"Hello, how are you?",greeting,greeting,True,True,True
1,Hey there! Good morning!,greeting,greeting,True,True,True
2,Hi! How's everything going?,greeting,greeting,True,True,True
3,"Good evening, how are you doing?",greeting,greeting,True,True,True
4,Howdy! Nice to meet you.,greeting,greeting,True,True,True
5,Good afternoon! Hope you're having a great day.,greeting,greeting,True,True,True
6,"Hey doc, how are you?",greeting,greeting,True,True,True
7,"Hi there, hope you're well!",greeting,greeting,True,True,True
8,Okay! Goodbye!,goodbye,goodbye,True,True,True
9,"See you later, take care!",goodbye,goodbye,True,True,True


### We will be proceeding with **LlaMA** and **Fewshots prompting** in our mental chatbot app